In [2]:
import pandas as pd

In [ ]:
# Loading the cleaned dataset
df = pd.read_csv("../data/cleaned/sales_data_cleaned.csv")

In [4]:
df.head()

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Unit_Price,Discount,Payment_Method,Returned,Sales
0,1001,2026-01-03,C001,Laptop,Electronics,North,1,65000,0.05,UPI,No,65000
1,1002,2026-01-04,C002,Mouse,Accessories,South,2,800,0.00,Credit Card,No,1600
2,1003,2026-01-05,C003,Keyboard,Accessories,West,1,1800,0.10,UPI,No,1800
3,1004,2026-01-06,C004,Monitor,Electronics,East,1,15000,0.05,Debit Card,No,15000
4,1005,2026-01-08,C005,Headphones,Accessories,North,2,2500,0.00,UPI,Yes,5000


In [5]:
#Convert the date
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

In [6]:
# We can extract information from a datetime column using .dt
df["Month"] = df["Order_Date"].dt.month

In [ ]:
# it also let us extract things such as :-
df["Order_Date"].dt.year
df["Order_Date"].dt.month
df["Order_Date"].dt.day
df["Order_Date"].dt.day_name()

### Extracting Date Components

The `.dt` accessor allows us to extract useful information from datetime columns.

- `.dt.year` → extracts the year
- `.dt.month` → extracts the month
- `.dt.day` → extracts the day
- `.dt.day_name()` → extracts the weekday name

In [ ]:
# Improved Sales calculation
# 1) Gross Sale
df["Gross_Sales"] = df["Quantity"] * df["Unit_Price"]


In [ ]:
# Discount _ Amount
df["Discount_Amount"] = df["Gross_Sales"] * df["Discount"]

In [ ]:
# Net Sales
df["Net_Sales"] = df["Gross_Sales"] - df["Discount_Amount"]

In [10]:
# Now inspect
df[
    [
        "Order_ID",
        "Product",
        "Quantity",
        "Unit_Price",
        "Discount",
        "Gross_Sales",
        "Discount_Amount",
        "Net_Sales"
    ]
]

,Order_ID,Product,Quantity,Unit_Price,Discount,Gross_Sales,Discount_Amount,Net_Sales
0,1001,Laptop,1,65000,0.05,65000,3250.0,61750.0
1,1002,Mouse,2,800,0.00,1600,0.0,1600.0
2,1003,Keyboard,1,1800,0.10,1800,180.0,1620.0
3,1004,Monitor,1,15000,0.05,15000,750.0,14250.0
4,1005,Headphones,2,2500,0.00,5000,0.0,5000.0
5,1007,Mouse,3,750,0.00,2250,0.0,2250.0
6,1008,Monitor,2,14000,0.10,28000,2800.0,25200.0
7,1010,Headphones,1,2800,0.00,2800,0.0,2800.0
8,1011,Laptop,1,68000,0.05,68000,3400.0,64600.0
9,1012,Monitor,1,15500,0.00,15500,0.0,15500.0


In [11]:
# Business Question
# Ques 1) Which products generate the most revenue?
product_performance = (
    df.groupby("Product")["Net_Sales"]
      .sum()
      .sort_values(ascending=False)
)

product_performance

Product
Laptop        126350.0
Monitor        54950.0
Headphones     12480.0
Mouse           6510.0
Keyboard        3720.0
Name: Net_Sales, dtype: float64

## Business Question 1

### Which products generate the most revenue?

Answer:
- Top product: Laptop
- Net sales: ₹ 126350.0
- Lowest-performing product: Keyboard
- Net sales: ₹ 3720.0

In [12]:
# Ques 2 ) Which category performs better?
category_performance = (
    df.groupby("Category")["Net_Sales"]
      .sum()
      .sort_values(ascending=False)
)

category_performance

Category
Electronics    181300.0
Accessories     22710.0
Name: Net_Sales, dtype: float64

In [26]:
df.groupby("Region")["Net_Sales"].agg(["sum", "mean", "count"])

,sum,mean,count
Region,,,
East,42110.0,14036.666667,3
North,82250.0,27416.666667,3
South,6500.0,2166.666667,3
West,73150.0,18287.500000,4


In [14]:
# Average Order Value
average_order_value = df["Net_Sales"].mean()

print("Average Order Value: ₹", round(average_order_value, 2))

Average Order Value: ₹ 15693.08


In [15]:
# Question 4) — Discounts
# Are discounts reducing our revenue significantly?
total_gross_sales = df["Gross_Sales"].sum()
total_discount = df["Discount_Amount"].sum()
total_net_sales = df["Net_Sales"].sum()

print("Gross Sales: ₹", total_gross_sales)
print("Discount Given: ₹", total_discount)
print("Net Sales: ₹", total_net_sales)


Gross Sales: ₹ 215050
Discount Given: ₹ 11040.0
Net Sales: ₹ 204010.0


In [16]:
# Discount Percentage 
discount_percentage = (
    total_discount / total_gross_sales
) * 100

print("Overall Discount Percentage:",
      round(discount_percentage, 2), "%")

Overall Discount Percentage: 5.13 %


In [17]:
# Question 5) — Returns
# Our dataset contains: Returned
# Let's investigate it.

In [18]:
return_counts = df["Returned"].value_counts()

return_counts

Returned
No     11
Yes     2
Name: count, dtype: int64

In [19]:
return_percentage = (
    df["Returned"].eq("Yes").mean() * 100
)

print(
    "Return Rate:",
    round(return_percentage, 2),
    "%"
)

Return Rate: 15.38 %


In [27]:
# Investigating returned products
returned_products = (
    df[df["Returned"] == "Yes"]
      .groupby("Product")["Order_ID"]
      .count()
      .sort_values(ascending=False)
)

returned_products

Product
Headphones    2
Name: Order_ID, dtype: int64

In [25]:
# Monthly performance 
df["Month_Name"] = pd.to_datetime(
    df["Month"].astype(str),
    format="%m"
).dt.month_name()
monthly_sales = (
    df.groupby("Month_Name")["Net_Sales"]
      .sum()
)
monthly_sales

Month_Name
January    204010.0
Name: Net_Sales, dtype: float64

## 📊 Day 3 Business Insights

### Revenue
- Highest revenue-generating product: Laptop
- Lowest revenue-generating product: Headphone
- Highest-performing category: Electronics
- Highest-performing region: North

### Customer/Order Metrics
- Average Order Value: ₹ 15693.08

### Discounts
- Gross Sales: ₹ 215050
- Total Discount Given: ₹ 11040
- Net Sales: ₹ 204010.0
- Overall Discount Percentage: 5.3%

### Returns
- Overall Return Rate: 15.38%
- Product with most returned orders: Headphone

### Key Observation

### Key Observation

Electronics generated the highest revenue, while Accessories had the lowest. This suggests that Electronics is currently the strongest revenue driver and may deserve greater inventory and marketing attention

In [28]:
# Saving 
df.to_csv(
    "../data/cleaned/sales_data_cleaned.csv",
    index=False
)